# 🎯 [OPS04_M06] 방공 지휘관 결심 지원을 위한 실시간 포화경보 우선순위 대시보드 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m06_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M06 |
| **미션 주제** | 방공 지휘관 결심 지원을 위한 실시간 포화경보 우선순위 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **핵심 산출물** | `src/ui/ops04_dashboard.py (또는 웹 UI 컴포넌트)` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 방공 지휘관 결심 지원을 위한 실시간 포화경보 우선순위 대시보드 구현
**소요시간**: 약 2.5시간 | **이번 산출물**: `src/ui/ops04_dashboard.py (또는 웹 UI 컴포넌트)`

### 📋 미션 개요
**포화 공격 상황에서 최우선 확인 표적(P1/P2/스푸핑 의심), 도달시간 분포, 대응 자원 잔여 현황을 한 화면에 직관적으로 표출하는 지휘관용 대시보드를 구현합니다.**

- **배경 및 필요성:** 수십~수백 개의 항적이 동시에 유입되는 상황에서 원시 로그나 텍스트 표만으로는 지휘관이 수 초 내에 우선 위협을 식별하기 어려워, 인지 부하를 낮추는 전술 상황판 UI가 필수적입니다.
- **핵심 과업:** 가상 방어지역 지도 위에 P1(180초 이내)·P2(600초 이내)·스푸핑 의심 항적을 색상으로 구분 표시하고 도달시간 분포 및 자원 현황을 고정 배치한 뒤, 1초 주기 10회 갱신 시뮬레이션으로 처리시간과 조회 응답시간을 측정합니다.
- **학습 목표 (습득 역량):** 지휘 결심 지원용 시각화 정보 구조(IA) 설계 능력, 시간 임계값 기반 시각적 주의 집중(Color-coding) 구현 역량, 실시간 주기 갱신 환경의 UI 응답성 계측 능력을 습득합니다.
- **검증 및 산출물:** 화면 갱신 주기 1초 이내, UI 응답성 100ms 이내를 충족하는 대시보드 모듈(`src/ui/ops04_dashboard.py` 또는 웹 UI 컴포넌트)을 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 포화 공격 시 지휘관의 인지 과부하를 줄이는 전술 상황판 정보 구조(IA)를 설계하고, 우선순위 색상 부호화 및 실시간 주기 갱신 UI 응답성을 검증하는 대시보드 구현 역량을 체득합니다.

**[핵심 과업]** 가상 방어지역 지도에 항적을 P1(180초 이내)·P2(600초 이내)·스푸핑 의심 색상으로 표출하고 도달시간 분포와 가상 대응 자원 잔여 현황을 고정 배치한 뒤, 1초 주기 갱신 10회 모의로 처리시간과 조회 응답시간을 측정합니다.

- **지휘 결심 지원용 시각화 정보 구조 설계 역량:** 다수 항적이 동시 유입될 때 지도·권고 순위 목록·도달시간 분포·자원 현황을 시선 이동 없이 한 화면에 고정 배치해 인지 부하를 낮추는 전술 UI 설계 원리를 익힙니다.
- **시간 임계값 기반 시각적 주의 집중 구현 능력:** TTI 기준(P1 180초 이내, P2 600초 이내)과 스푸핑 의심 상태를 색상과 표식으로 명확히 구분해 지휘관이 최우선 표적을 즉시 식별하도록 돕는 시각화 기술을 습득합니다.
- **실시간 스트림 갱신 및 UI 응답성 계측 역량:** 1초 주기 10회 연속 갱신 시뮬레이션에서 갱신 처리시간(1초 이내)과 목록 조회 응답시간(100ms 이내)을 프로파일링해 전술 상황판의 실시간성을 보증하는 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
포화 공격 상황에서 가장 먼저 요격해야 할 표적 목록과 방공망 대응 현황을 직관적으로 표출하는 포화경보 우선순위 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **포화 상황에서 먼저 확인할 표적과 대응 자원 현황을 한 화면에 보여 주는 지휘관용 대시보드를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 방어지역 지도에 항적을 우선순위 색(P1 180초 이내, P2 600초 이내, 스푸핑 의심)으로 표시하고, 도달시간 분포와 가상 대응 자원 잔여 현황을 고정 배치 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/ops04_dashboard.py (또는 웹 UI 컴포넌트)` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 6 코드를 실행해 갱신 처리시간과 조회 응답시간을 확인한다.
#### 이 단계의 역할
미션 6의 대시보드 `refresh()` 모듈을 실행해 1초 주기 갱신 10회 동안의 갱신 처리시간과 목록 조회 응답시간을 측정합니다.

#### 해야 할 일

- 가상 방어지역 지도, 우선순위 목록(P1/P2/스푸핑 의심), 도달시간 분포, 가상 대응 자원 현황 4대 UI 영역 구성을 확인한다.
- AI를 Co-pilot으로 활용해 1초 주기 10회 연속 갱신 시 평균·최대 처리시간(ms)과 조회 응답시간을 계측하는 벤치마크 루프를 점검한다.
- 화면 갱신 주기 1초 이내(평균 9.4 ms / 최대 10.9 ms)와 UI 응답성 100ms 이내 달성 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "지휘관용 포화경보 대시보드 refresh() 함수를 1초 주기로 10회 모의 갱신하면서 각 회차의 데이터 갱신 처리시간(ms)과 우선순위 목록 조회 응답시간(ms)의 평균/최대값을 출력하는 성능 검증 코드를 검토해줘."

  - **AI 결과 검증 포인트:** AI가 작성한 UI 코드가 매 갱신 주기마다 전체 모델 학습을 반복해 처리시간이 100ms를 넘기지 않는지, 사전 계산된 상태 조회 구조를 쓰는지 확인합니다.

#### 핵심 확인 지표

> 갱신 처리 수십 ms 이내(평균 9.4 ms / 최대 10.9 ms), 조회 응답 수 ms 이내, P1 13건·P2 20건·스푸핑 의심 6건

#### 학습자가 제출해야 하는 내용

- 미션 6 대시보드 갱신·응답시간 계측 로그
- AI 프롬프트 질의 및 UI 성능 병목 점검 기록

#### Step 2. 대시보드 지도에서 P1(빨강) 표적과 지향 자산을 확인한다.
#### 이 단계의 역할
대시보드 지도 위에서 P1(180초 이내, 빨강) 최우선 표적과 해당 표적이 지향하는 가상 방어자산의 매핑 정확성을 검증합니다.

#### 해야 할 일

- 시각화된 가상 방어지역 지도에서 P1(180초 이내), P2(600초 이내), 스푸핑 의심 항적의 색상·마커 구분을 확인한다.
- AI에게 지도상 P1 표적 13건의 좌표·진행 방향·지향 자산(`A1~A4`)을 텍스트 표로 교차 출력해 시각화 누락이 없는지 검증하는 코드를 요청한다.
- 지도상의 P1 표적 위치와 우선순위 목록 표의 지향 자산 일치 여부를 대조한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "대시보드에 표시되는 P1(TTI 
#### 핵심 확인 지표

> P1 13건·P2 20건·스푸핑 의심 6건 시각화 및 지향 자산 1:1 매핑 일치

#### 학습자가 제출해야 하는 내용

- 대시보드 지도 캡처/출력 결과 및 P1 표적-지향 자산 대조표
- AI 검증 코드 및 시각 가독성 점검 기록

#### Step 3. 도달시간 분포와 자원 현황 영역이 한눈에 읽히는지 동료와 평가한다.
#### 이 단계의 역할
도달시간(TTI) 분포와 가상 대응 자원(`RES`) 잔여 현황 패널이 지휘관의 인지 부하를 줄이도록 한눈에 읽히는지 평가·개선합니다.

#### 해야 할 일

- 대시보드 내 4개 핵심 영역(지도, 권고 순위 목록, TTI 분포, 자원 잔여 현황)의 고정 배치와 색상 대비를 점검한다.
- AI에게 야간 작전 상황이나 포화 경보 집중 시 인지 부하를 줄이기 위한 레이아웃/레이블 개선 체크리스트를 요청한다.
- 동료 평가 또는 체크리스트 대조를 통해 가독성 개선 사항과 UI 응답성(100ms 이내)을 최종 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "포화경보 대시보드의 4대 패널(지도, 권고 목록, TTI 분포, 자원 현황)에 대해 방공 지휘관 관점에서 인지 부하를 줄이기 위한 색각 안전성, 폰트 크기, 경고 강조 개선 포인트 5가지를 점검 체크리스트로 만들어줘."

  - **AI 결과 검증 포인트:** AI가 제안한 UI 개선안을 적용할 때 1초 주기 고정 레이아웃이 흔들리거나(레이아웃 시프트) 불필요한 애니메이션으로 응답성이 100ms를 초과하지 않도록 통제합니다.

#### 핵심 확인 지표

> 주요 시각화 컴포넌트 4종 정상 렌더링, 고정 레이아웃 인지 가독성 점검 완료

#### 학습자가 제출해야 하는 내용

- 대시보드 가독성 평가 메모 및 UI 개선 적용 결과
- AI 제안 UI 개선안 채택/기각 검토 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
P1 긴급 기준을 `180초`에서 `120초`로 축소했을 때 지휘관이 최우선으로 확인해야 할 표적 수와 경보 밀도 변화를 실험합니다.

#### 해야 할 일

- P1 기준 임계값을 `180초`에서 `120초`로 변경해 대시보드 분류를 재실행한다.
- AI에게 P1 기준 단축에 따른 P1/P2 표적 수 변화와 요격 대응 여유시간(180초 vs 120초) 변화 분석을 요청한다.
- 변경 전후 P1 표적 수와 지휘관 결심 여유시간 관점의 장단점을 비교 메모로 작성한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "대시보드의 P1 기준을 180초에서 120초로 바꿀 때 P1과 P2 표적 수 변화를 비교하고, 방공 지휘관의 상황 인지 부담 감소 vs 대응 준비시간 단축 관점에서 장단점을 분석해줘."

  - **AI 결과 검증 포인트:** 120초로 줄여 화면의 빨간 표적 수가 줄더라도 실제 방어자산 도달까지 남은 물리적 대응 시간이 60초 감소한다는 전술적 리스크를 명확히 짚었는지 확인합니다.

#### 핵심 확인 지표

> P1 기준 180초 vs 120초 표적 수 변화 및 대응 여유시간 분석 완료

#### 학습자가 제출해야 하는 내용

- P1 임계값 변경 전후 비교 분석 메모 및 최종 시각화 결과
- AI 질의 프롬프트 및 전술적 트레이드오프 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 전술 작전 상황판으로서의 직관성 및 인지 부하 감소도

#### 미션 PASS 조건
- 정량 기준: 화면 갱신 주기 1초 이내, UI 응답성 100ms 이내
- 진행 기준: 주요 지표 시각화 컴포넌트 4종 이상 렌더링 및 실시간 데이터 연동
- 검증 방법: 사용자 인터랙션 테스트 및 모의 데이터 스트림 렌더링 검증
- AI 검증 기준: 대시보드 4대 패널의 1초 주기 갱신 성능(100ms 이내 응답), P1 표적 매핑 및 가독성 개선을 AI 검증 포인트로 점검 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
# [TODO : 학생 실습 6] 핵심 함수의 TODO 영역을 AI Co-pilot과 함께 구현하고 검증하세요.
"""방공 지휘관 결심 지원용 포화경보 우선순위 대시보드를 구현한다. 가상 방어지역 지도에 표적 항적과 우선순위를, 옆에 권고 우선순위 목록·도달시간 분포·가상 대응 자원 현황을 표시하고 1초 주기 갱신 처리시간과 조회 응답시간을 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

clf = train_classifier()
raid = make_raid(9, 100, spoof=6)
RES = {"R1": [6, 6], "R2": [4, 4], "R3": [8, 8]}       # [잔여, 최초] 가상 대응 자원


def refresh(df):
    """[TODO : 학생 실습 6] advanced_rank 결과를 좌표와 병합해 P1(<=180s), P2, SPOOF, P3 등급(prio)을 부여하세요."""
    # 힌트: r = advanced_rank(df, clf).merge(df[["id", "x", "y", "vx", "vy"]], on="id") 후 prio 컬럼 생성
    pass  # <<< 여기에 refresh() 로직을 작성한 뒤 pass를 제거하세요 >>>


# 1) 1초 주기 갱신 10회 모의: 갱신 처리시간과 조회(필터·정렬) 응답시간 측정
upd, qry = [], []
for k in range(10):
    t0 = time.perf_counter(); board = refresh(raid); upd.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); view = board[board["prio"].isin(["P1", "P2"])].sort_values("tti_s").head(12); qry.append((time.perf_counter() - t0) * 1000)
    raid = raid.assign(x=raid["x"] + raid["vx"], y=raid["y"] + raid["vy"])
    raid = raid[np.hypot(raid["x"], raid["y"]) > 0.5].reset_index(drop=True)   # 도달 처리된 항적 제거(가상)
print(f"갱신 처리시간 평균 {np.mean(upd):.1f} ms / 최대 {np.max(upd):.1f} ms (목표: 갱신 주기 1초 이내) -> {'달성' if max(upd) < 1000 else '미달'}")
print(f"조회 응답시간 최대 {np.max(qry):.2f} ms (목표 100 ms 이내) -> {'달성' if max(qry) < 100 else '미달'}")
cnt = board["prio"].value_counts()
print(f"현재 표시: P1(180초 이내) {cnt.get('P1', 0)}건, P2(600초 이내) {cnt.get('P2', 0)}건, 스푸핑 의심 {cnt.get('SPOOF', 0)}건, 기타 {cnt.get('P3', 0)}건")
print("\n[권고 우선순위 목록] (P1·P2, 도달 예상 순)")
print(view[["id", "prio", "asset", "tti_s", "p_threat"]].round({"tti_s": 0, "p_threat": 2}).rename(
    columns={"id": "표적", "prio": "등급", "asset": "지향자산", "tti_s": "도달예상(s)", "p_threat": "위협유사도"}).to_string(index=False))

# 2) 대시보드(어두운 배경, P1은 빨강 굵게, 정보는 4개 영역으로 고정 배치해 인지 부하를 줄인다)
plt.style.use("dark_background")
fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(2, 3)
a = fig.add_subplot(gs[:, :2])
col = {"P1": "red", "P2": "orange", "P3": "gray", "SPOOF": "magenta"}
for p, g in board.groupby("prio"):
    a.quiver(g["x"], g["y"], g["vx"], g["vy"], color=col[p], scale=3, width=0.003, label=f"{p} ({len(g)})")
a.scatter(ASSETS["x"], ASSETS["y"], marker="^", s=150, c="cyan")
for r in ASSETS.itertuples():
    a.text(r.x + 0.8, r.y + 0.8, f"{r.asset} v={r.value}", color="cyan", fontsize=9)
for r in board[board["prio"] == "P1"].itertuples():
    a.text(r.x, r.y, r.id[-3:], color="red", fontsize=8, fontweight="bold")
a.set_xlim(-60, 60); a.set_ylim(-60, 60); a.set_aspect("equal"); a.grid(alpha=0.2); a.legend(loc="lower left")
a.set_title("Saturation alert picture (virtual area)"); a.set_xlabel("East (km)"); a.set_ylabel("North (km)")
b = fig.add_subplot(gs[0, 2])
urg = board[board["urgent_pred"]]
b.hist(urg["tti_s"], bins=np.arange(0, 660, 60), color="orange"); b.axvline(180, c="red", ls="--")
b.set_title("Urgent tracks by TTI (s)")
c = fig.add_subplot(gs[1, 2])
c.barh(list(RES), [v[1] for v in RES.values()], color="dimgray", label="initial")
c.barh(list(RES), [v[0] for v in RES.values()], color="lime", label="remaining")
c.set_title("Virtual response resources"); c.legend(fontsize=8)
plt.suptitle("OPS04 dashboard - recommendations only, commander approval required (training data)")
plt.tight_layout(); plt.show()
print("\n대시보드의 순위는 권고 정보이며, 대응 여부는 미션 7의 지휘관 승인 절차와 교전수칙·지휘계통에 따른다.")

<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
갱신 처리시간 평균 9.4 ms / 최대 10.9 ms (목표: 갱신 주기 1초 이내) -> 달성
조회 응답시간 최대 1.64 ms (목표 100 ms 이내) -> 달성
현재 표시: P1(180초 이내) 13건, P2(600초 이내) 20건, 스푸핑 의심 6건, 기타 67건

[권고 우선순위 목록] (P1·P2, 도달 예상 순)
    표적 등급 지향자산  도달예상(s)  위협유사도
T09027 P1   A3      0.0   1.00
T09017 P1   A4     70.0   1.00
T09072 P1   A1     76.0   1.00
T09047 P1   A2     78.0   0.99
T09038 P1   A3     85.0   0.96
T09069 P1   A4     97.0   1.00
T09049 P1   A3    136.0   0.99
T09023 P1   A4    147.0   1.00
T09092 P1   A3    149.0   0.97
T09006 P1   A1    155.0   0.93
T09089 P1   A2    160.0   1.00
T09026 P1   A2    167.0   1.00

대시보드의 순위는 권고 정보이며, 대응 여부는 미션 7의 지휘관 승인 절차와 교전수칙·지휘계통에 따른다.
```
</details>
